# Leather Defect Analysis — CLIP + Moondream/LLaVA
### Seminar paper: Application of Multimodal Models for Visual Content Analysis

---
**Run order:** Execute cells sequentially from top to bottom.  
**Prerequisites:** Ollama must be running in the background (system tray icon) before executing LLaVA/Moondream cells.

## Cell 0 — Library Installation
Run only once on first use.

In [ ]:
import sys
!{sys.executable} -m pip install openai-clip --quiet
!{sys.executable} -m pip install torch torchvision --quiet
!{sys.executable} -m pip install pillow numpy pandas matplotlib seaborn scikit-learn tqdm openpyxl requests --quiet
print('\n✅ All libraries installed!')

## Cell 1 — Imports and Path Configuration
⚠️ **Adjust all paths marked with `...` to match your local folder structure before running.**

In [ ]:
import os
import json
import base64
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# ► ADJUST THESE PATHS TO YOUR LOCAL SETUP
# ============================================================
BOLD_FOLDER = r"C:/Users/.../Integrated-Database-for-Systematic-Analysis-of-Defects-in-Tanned-Leather/Original samples/Bold"
ELBA_FOLDER = r"C:/Users/.../Integrated-Database-for-Systematic-Analysis-of-Defects-in-Tanned-Leather/Original samples/Elba"

BOLD_EXCEL  = r"C:/Users/.../Integrated-Database-for-Systematic-Analysis-of-Defects-in-Tanned-Leather/Annotated samples/Bold_defects.xlsx"
ELBA_EXCEL  = r"C:/Users/.../Integrated-Database-for-Systematic-Analysis-of-Defects-in-Tanned-Leather/Annotated samples/Elba_defects.xlsx"

# Folder where results will be saved (created automatically if it doesn't exist)
RESULTS_DIR = r"C:/Users/.../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Ollama model to use for VQA (change as needed)
OLLAMA_MODEL = "moondream"   # options: "moondream", "llava:7b-v1.6", "llava:13b"

print(f"✅ Configuration loaded")
print(f"   BOLD folder : {'✅ exists' if os.path.exists(BOLD_FOLDER) else '❌ NOT FOUND — check path!'}")
print(f"   ELBA folder : {'✅ exists' if os.path.exists(ELBA_FOLDER) else '❌ NOT FOUND — check path!'}")
print(f"   Results dir : {RESULTS_DIR}")

## Cell 2 — Load Excel Annotation Tables and Build Ground Truth
Reads defect annotation tables from Excel files and constructs the ground truth DataFrame.

In [ ]:
def load_defects_excel(filepath, leather_type):
    """Reads a defect annotation Excel file and returns a clean DataFrame."""
    df_raw = pd.read_excel(filepath, header=0)
    
    rows = []
    for _, row in df_raw.iterrows():
        values = [str(v).strip() for v in row.values if pd.notna(v) and str(v).strip() != '']
        # Skip comment/legend rows (rows not starting with leather type code)
        if not values or values[0] not in ['BO', 'EL']:
            continue
        ltype = values[0]
        try:
            label = int(values[1])
        except:
            continue
        defects = [v.lower().strip() for v in values[2:]
                   if v not in ['BO', 'EL'] and not v.startswith('Note')
                   and not v.startswith('COLOR') and len(v) < 40]
        defect_str  = '; '.join(defects) if defects else ''
        defect_free = any('defect free' in d for d in defects)
        rows.append({
            'leather_type'  : ltype,
            'label'         : label,
            'defect_free'   : defect_free,
            'defect_present': not defect_free,
            'defects_raw'   : defect_str,
            'filename'      : f"{ltype}{label:03d}.tif"
        })
    return pd.DataFrame(rows)

df_bold = load_defects_excel(BOLD_EXCEL, 'BO')
df_elba = load_defects_excel(ELBA_EXCEL, 'EL')
df_gt   = pd.concat([df_bold, df_elba], ignore_index=True)

print(f"✅ Ground truth loaded:")
print(f"   Bold samples : {len(df_bold)} (defective: {df_bold['defect_present'].sum()}, clean: {(~df_bold['defect_present']).sum()})")
print(f"   Elba samples : {len(df_elba)} (defective: {df_elba['defect_present'].sum()}, clean: {(~df_elba['defect_present']).sum()})")
print(f"   TOTAL        : {len(df_gt)} samples")
print()
df_gt.head(5)

## Cell 3 — Verify Image Availability
Checks that image files exist on disk and builds working DataFrame with image paths.

In [ ]:
def find_image(folder, filename):
    """Searches for an image in a folder — tries .tif and .jpg variants."""
    for ext in ['.tif', '.TIF', '.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']:
        base = filename.replace('.tif', '').replace('.TIF', '')
        path = os.path.join(folder, base + ext)
        if os.path.exists(path):
            return path
    return None

def get_folder(ltype):
    return BOLD_FOLDER if ltype == 'BO' else ELBA_FOLDER

df_gt['image_path'] = df_gt.apply(
    lambda r: find_image(get_folder(r['leather_type']), r['filename']), axis=1
)

found    = df_gt['image_path'].notna().sum()
notfound = df_gt['image_path'].isna().sum()

print(f"✅ Images found: {found}/{len(df_gt)}")
if notfound > 0:
    print(f"⚠️  Not found ({notfound}):")
    print(df_gt[df_gt['image_path'].isna()]['filename'].tolist())

# Work only with found images
df_work = df_gt[df_gt['image_path'].notna()].copy().reset_index(drop=True)
print(f"\n   Working with {len(df_work)} samples.")

## Cell 3b — Export Ground Truth Labels to CSV

In [ ]:
import os
import pandas as pd

# Save ground truth labels as CSV
labels_path = os.path.join(RESULTS_DIR, 'labels_ground_truth.csv')
df_gt[['filename', 'leather_type', 'label', 'defect_present', 'defects_raw']].to_csv(
    labels_path, index=False, encoding='utf-8-sig'
)
print(f"✅ Ground truth labels saved to: {labels_path}")
print(df_gt[['filename', 'leather_type', 'defect_present', 'defects_raw']].head(5))

## Cell 4 — TIF to JPG Conversion
Moondream and LLaVA models do not support .tif format directly.  
This cell converts all .tif images to .jpg before sending to Ollama models.  
⚠️ **Adjust source and destination paths below.**

In [ ]:
from PIL import Image
import os
from tqdm.notebook import tqdm

# ► Adjust these paths
SRC_TIF = r"C:/Users/.../slike_tif"   # folder with original .tif images
DST_JPG = r"C:/Users/.../slike_jpg"   # destination folder for .jpg images
os.makedirs(DST_JPG, exist_ok=True)

tif_files = [f for f in os.listdir(SRC_TIF) if f.lower().endswith('.tif')]
print(f"Found {len(tif_files)} TIF files to convert")

errors = []
for fname in tqdm(tif_files, desc="Converting"):
    try:
        img = Image.open(os.path.join(SRC_TIF, fname)).convert('RGB')
        jpg_name = os.path.splitext(fname)[0] + '.jpg'
        img.save(os.path.join(DST_JPG, jpg_name), 'JPEG', quality=95)
    except Exception as e:
        errors.append(f"{fname}: {e}")

print(f"\n✅ Converted: {len(tif_files) - len(errors)}/{len(tif_files)}")
if errors:
    print(f"❌ Errors: {errors}")

## Cell 5 — Check Ollama Availability
Verifies that the Ollama service is running and that required models are installed.

In [ ]:
def check_ollama():
    """Check whether Ollama is running and the target model is available."""
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=5)
        models = [m['name'] for m in r.json().get('models', [])]
        print(f"✅ Ollama is running!")
        print(f"   Available models: {models}")
        if any(OLLAMA_MODEL in m for m in models):
            print(f"   ✅ Model '{OLLAMA_MODEL}' is ready!")
        else:
            print(f"   ❌ Model '{OLLAMA_MODEL}' NOT found!")
            print(f"      Open Anaconda Prompt and run: ollama pull {OLLAMA_MODEL}")
    except:
        print("❌ Ollama is NOT running!")
        print("   Open the Ollama application from the Start menu,")
        print("   or run in Anaconda Prompt: ollama serve")

check_ollama()

## Cell 6 — Model 1: CLIP Zero-Shot Classification
⏱️ **Estimated time: ~10–30 seconds for all 70 samples (very fast)**

CLIP performs zero-shot classification by measuring cosine similarity between image embeddings and a set of text prompt descriptions for each defect category.

In [ ]:
import clip
import torch

print("Loading CLIP model... (may take 1–2 minutes on first run)")
device = "cuda" if torch.cuda.is_available() else "cpu"
model_clip, preprocess_clip = clip.load("ViT-B/32", device=device)
print(f"✅ CLIP loaded on: {device}")

# Text prompt descriptions for each defect category
DEFECT_PROMPTS = [
    "a leather surface with visible color defects or discoloration",
    "a leather surface with grain off or damaged surface texture",
    "a leather surface with folding marks or creases",
    "a leather surface with pin holes or small holes",
    "a leather surface with cuts or scratches or scars",
    "a leather surface with wrinkles or neck wrinkles",
    "a leather surface with loose grain or growth marks",
    "a clean leather surface without any visible defects"
]

DEFECT_LABELS = [
    "color", "grain_off", "folding_marks", "pin_holes",
    "cuts_scars", "wrinkles", "loose_grain", "defect_free"
]

# Encode all text prompts once (more efficient)
text_tokens = clip.tokenize(DEFECT_PROMPTS).to(device)
with torch.no_grad():
    text_features = model_clip.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print(f"✅ {len(DEFECT_PROMPTS)} text prompts encoded")

def classify_with_clip(image_path):
    """Classifies a single image using CLIP zero-shot classification."""
    try:
        img = Image.open(image_path).convert('RGB')
        img_tensor = preprocess_clip(img).unsqueeze(0).to(device)
        with torch.no_grad():
            img_features = model_clip.encode_image(img_tensor)
            img_features = img_features / img_features.norm(dim=-1, keepdim=True)
            similarities = (img_features @ text_features.T).squeeze(0)
            probs        = similarities.softmax(dim=-1).cpu().numpy()
        top_idx   = int(np.argmax(probs))
        top_label = DEFECT_LABELS[top_idx]
        top_prob  = float(probs[top_idx])
        all_probs = {DEFECT_LABELS[i]: float(probs[i]) for i in range(len(DEFECT_LABELS))}
        return top_label, top_prob, all_probs
    except Exception as e:
        return 'error', 0.0, {}

# Run CLIP on all samples
clip_results = []
print("\nRunning CLIP classification...")

for _, row in tqdm(df_work.iterrows(), total=len(df_work), desc="CLIP"):
    pred_label, pred_prob, all_probs = classify_with_clip(row['image_path'])
    clip_results.append({
        'filename'             : row['filename'],
        'leather_type'         : row['leather_type'],
        'gt_defect_present'    : row['defect_present'],
        'gt_defects'           : row['defects_raw'],
        'clip_top_label'       : pred_label,
        'clip_top_prob'        : pred_prob,
        'clip_defect_predicted': pred_label != 'defect_free',
        **{f'prob_{k}': v for k, v in all_probs.items()}
    })

df_clip = pd.DataFrame(clip_results)

# Save results
clip_csv = os.path.join(RESULTS_DIR, 'clip_results.csv')
df_clip.to_csv(clip_csv, index=False, encoding='utf-8-sig')
print(f"\n✅ CLIP done! Results saved to: {clip_csv}")
df_clip[['filename', 'gt_defect_present', 'clip_top_label', 'clip_defect_predicted', 'clip_top_prob']].head(10)

## Cell 7 — CLIP Metrics and Visualizations

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

y_true = df_clip['gt_defect_present'].astype(int)
y_pred = df_clip['clip_defect_predicted'].astype(int)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)

print("="*45)
print("  CLIP — Binary Classification Results")
print("="*45)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)")
print(f"  Precision : {prec:.4f}  ({prec*100:.1f}%)")
print(f"  Recall    : {rec:.4f}  ({rec*100:.1f}%)")
print(f"  F1-Score  : {f1:.4f}  ({f1*100:.1f}%)")
print("="*45)
print()
print("Detailed report:")
print(classification_report(y_true, y_pred,
      target_names=['defect_free', 'defect_present']))

# Confusion Matrix + distribution by leather type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Clean', 'Pred: Defect'],
            yticklabels=['True: Clean', 'True: Defect'])
axes[0].set_title('CLIP — Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Ground Truth')
axes[0].set_xlabel('CLIP Prediction')

# Distribution by leather type
summary = df_clip.groupby('leather_type').agg(
    GT_Defect    = ('gt_defect_present', 'sum'),
    GT_Clean     = ('gt_defect_present', lambda x: (~x.astype(bool)).sum()),
    CLIP_Defect  = ('clip_defect_predicted', 'sum'),
    CLIP_Clean   = ('clip_defect_predicted', lambda x: (~x.astype(bool)).sum())
).reset_index()

x = np.arange(len(summary))
w = 0.2
axes[1].bar(x - 1.5*w, summary['GT_Defect'],  w, label='GT: Defect',  color='#F44336', alpha=0.85)
axes[1].bar(x - 0.5*w, summary['GT_Clean'],   w, label='GT: Clean',   color='#2196F3', alpha=0.85)
axes[1].bar(x + 0.5*w, summary['CLIP_Defect'],w, label='CLIP: Defect',color='#D32F2F', alpha=0.55, hatch='//')
axes[1].bar(x + 1.5*w, summary['CLIP_Clean'], w, label='CLIP: Clean', color='#1565C0', alpha=0.55, hatch='//')
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary['leather_type'])
axes[1].set_title('Distribution by Leather Type — Ground Truth vs CLIP', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Leather Type')
axes[1].set_ylabel('Number of Samples')
axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'clip_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Figure saved to: {RESULTS_DIR}")

## Cell 8 — CLIP: Example Correct and Incorrect Predictions

In [ ]:
df_clip_work = df_clip.copy()
df_clip_work['correct'] = (df_clip_work['gt_defect_present'] == df_clip_work['clip_defect_predicted'])

correct_samples   = df_clip_work[df_clip_work['correct']].head(4)
incorrect_samples = df_clip_work[~df_clip_work['correct']].head(4)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('CLIP — Prediction Examples\n(Top row: Correct | Bottom row: Incorrect)',
             fontsize=14, fontweight='bold')

def show_sample(ax, row, title_prefix):
    path = df_work[df_work['filename'] == row['filename']]['image_path'].values
    if len(path) == 0:
        ax.axis('off'); return
    try:
        img = Image.open(path[0]).convert('RGB')
        img.thumbnail((300, 300))
        ax.imshow(img)
    except:
        ax.axis('off'); return
    gt   = 'DEFECT' if row['gt_defect_present'] else 'CLEAN'
    pred = 'DEFECT' if row['clip_defect_predicted'] else 'CLEAN'
    color = 'green' if row['correct'] else 'red'
    ax.set_title(f"{row['filename']}\nGT: {gt} | PRED: {pred}\n({row['clip_top_label']})",
                 fontsize=8, color=color)
    ax.axis('off')

for i, (_, row) in enumerate(correct_samples.iterrows()):
    show_sample(axes[0][i], row, 'OK')
for i in range(len(correct_samples), 4):
    axes[0][i].axis('off')
for i, (_, row) in enumerate(incorrect_samples.iterrows()):
    show_sample(axes[1][i], row, 'ERROR')
for i in range(len(incorrect_samples), 4):
    axes[1][i].axis('off')

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'clip_examples.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Example predictions figure saved!")

## Cell 9 — Model 2: Moondream/LLaVA — Visual Question Answering

⚠️ **Read before running:**
- Results are saved every 10 images as a safety checkpoint
- **Do NOT interrupt** execution if possible — all results are kept in memory and written at checkpoints
- If the notebook crashes: re-run cells 1–5, then re-run this cell — it will start fresh (no resume logic to avoid CSV corruption)

**Estimated time:**
- `moondream`: ~20–40s per image → ~35–50 minutes for 70 images  
- `llava:7b-v1.6`: ~60–80s per image → ~70–90 minutes for 70 images

In [ ]:
# ► Adjust JPG folder path (converted from TIF in Cell 4)
JPG_FOLDER = r"C:/Users/.../slike_jpg"

OLLAMA_MODEL_LLAVA = "moondream"   # change to "llava:7b-v1.6" if desired
LLAVA_CSV = os.path.join(RESULTS_DIR, 'llava_moondream_results.csv')

def find_jpg(filename):
    base = os.path.splitext(filename)[0]
    path = os.path.join(JPG_FOLDER, base + '.jpg')
    return path if os.path.exists(path) else None

df_work['image_path_jpg'] = df_work['filename'].apply(find_jpg)
found_jpg = df_work['image_path_jpg'].notna().sum()
print(f"✅ JPG images found: {found_jpg}/{len(df_work)}")

PROMPT = """You are an expert in leather quality inspection.
Analyze this leather sample image carefully.

Answer these three questions:
1. DEFECT_PRESENT: Are there any visible defects on this leather surface? Answer only Yes or No.
2. DEFECT_TYPE: If yes, what type of defects do you see?
   Choose from: color, grain_off, folding_marks, pin_holes, cuts_scars, wrinkles, loose_grain, other.
   If no defects, write: none
3. SEVERITY: Rate the severity: None, Minor, or Major.

Format your answer exactly like this:
DEFECT_PRESENT: Yes/No
DEFECT_TYPE: type1, type2
SEVERITY: None/Minor/Major
DESCRIPTION: one sentence describing what you see"""

def query_ollama(image_path, prompt, model):
    """Sends an image to Ollama and returns the model response."""
    try:
        with open(image_path, 'rb') as f:
            img_b64 = base64.b64encode(f.read()).decode('utf-8')
        payload = {"model": model, "prompt": prompt, "images": [img_b64], "stream": False}
        r = requests.post('http://localhost:11434/api/generate', json=payload, timeout=300)
        if r.status_code == 200:
            return r.json().get('response', '').strip()
        else:
            return f"ERROR: HTTP {r.status_code}"
    except Exception as e:
        return f"ERROR: {str(e)}"

def parse_llava_response(response):
    """Parses structured model response into a dictionary."""
    result = {'defect_present': None, 'defect_type': None,
              'severity': None, 'description': None, 'raw_response': response}
    if response.startswith('ERROR'):
        return result
    for line in response.split('\n'):
        line = line.strip()
        if line.startswith('DEFECT_PRESENT:'):
            val = line.split(':', 1)[1].strip().lower()
            result['defect_present'] = 'yes' in val
        elif line.startswith('DEFECT_TYPE:'):
            result['defect_type'] = line.split(':', 1)[1].strip()
        elif line.startswith('SEVERITY:'):
            result['severity'] = line.split(':', 1)[1].strip()
        elif line.startswith('DESCRIPTION:'):
            result['description'] = line.split(':', 1)[1].strip()
    return result

# Collect all results in memory, write to CSV periodically (avoids header duplication bug)
all_results = []

print(f"🚀 Running {OLLAMA_MODEL_LLAVA} on {len(df_work)} images")
print(f"   Estimated time: ~{len(df_work)*35/60:.0f}–{len(df_work)*40/60:.0f} minutes")
print("   ⚠️ Do NOT interrupt — results are checkpointed every 10 images\n")

for idx, (_, row) in enumerate(tqdm(df_work.iterrows(), total=len(df_work), desc=OLLAMA_MODEL_LLAVA)):
    if row['image_path_jpg'] is None:
        continue
    start    = time.time()
    response = query_ollama(row['image_path_jpg'], PROMPT, OLLAMA_MODEL_LLAVA)
    elapsed  = time.time() - start
    parsed   = parse_llava_response(response)

    all_results.append({
        'filename'            : row['filename'],
        'leather_type'        : row['leather_type'],
        'gt_defect_present'   : row['defect_present'],
        'gt_defects'          : row['defects_raw'],
        'llava_defect_present': parsed['defect_present'],
        'llava_defect_type'   : parsed['defect_type'],
        'llava_severity'      : parsed['severity'],
        'llava_description'   : parsed['description'],
        'llava_raw_response'  : parsed['raw_response'],
        'processing_time_s'   : round(elapsed, 1)
    })

    # Safety checkpoint every 10 images
    if (idx + 1) % 10 == 0:
        pd.DataFrame(all_results).to_csv(LLAVA_CSV, index=False, encoding='utf-8-sig')
        print(f"   💾 [{idx+1}/{len(df_work)}] checkpoint saved")

# Final save
pd.DataFrame(all_results).to_csv(LLAVA_CSV, index=False, encoding='utf-8-sig')
print(f"\n✅ Done! {len(all_results)} results saved to: {LLAVA_CSV}")

## Cell 10 — Model Comparison: CLIP vs Moondream vs LLaVA-7B
Computes metrics for all models and generates comparison visualizations.

In [ ]:
# Load Moondream results
df_llava = pd.read_csv(os.path.join(RESULTS_DIR, 'llava_moondream_results.csv'))
df_llava = df_llava.drop_duplicates(subset='filename', keep='last')
df_llava_valid = df_llava[df_llava['llava_defect_present'].notna()].copy()
print(f"Moondream: {len(df_llava_valid)} successfully parsed responses out of {len(df_llava)}")

y_true_l = df_llava_valid['gt_defect_present'].astype(bool).astype(int)
y_pred_l = df_llava_valid['llava_defect_present'].astype(bool).astype(int)

acc_l  = accuracy_score(y_true_l, y_pred_l)
prec_l = precision_score(y_true_l, y_pred_l, zero_division=0)
rec_l  = recall_score(y_true_l, y_pred_l, zero_division=0)
f1_l   = f1_score(y_true_l, y_pred_l, zero_division=0)

print("\n" + "="*50)
print(f"  {'Metric':<15} {'CLIP':>10} {'Moondream':>12}")
print("="*50)
print(f"  {'Accuracy':<15} {acc*100:>9.1f}% {acc_l*100:>11.1f}%")
print(f"  {'Precision':<15} {prec*100:>9.1f}% {prec_l*100:>11.1f}%")
print(f"  {'Recall':<15} {rec*100:>9.1f}% {rec_l*100:>11.1f}%")
print(f"  {'F1-Score':<15} {f1*100:>9.1f}% {f1_l*100:>11.1f}%")
print("="*50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_l = confusion_matrix(y_true_l, y_pred_l)
sns.heatmap(cm_l, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Pred: Clean', 'Pred: Defect'],
            yticklabels=['True: Clean', 'True: Defect'])
axes[0].set_title('Moondream — Confusion Matrix', fontsize=13, fontweight='bold')

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1']
clip_vals  = [acc*100, prec*100, rec*100, f1*100]
llava_vals = [acc_l*100, prec_l*100, rec_l*100, f1_l*100]
x = np.arange(len(metrics_list))
w = 0.35
axes[1].bar(x - w/2, clip_vals,  w, label='CLIP',      color='#2196F3', alpha=0.85)
axes[1].bar(x + w/2, llava_vals, w, label='Moondream', color='#FF9800', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(metrics_list)
axes[1].set_ylim(0, 110)
axes[1].set_ylabel('Value (%)')
axes[1].set_title('CLIP vs Moondream — Metric Comparison', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Comparison figure saved!")

## Cell 11 — LLaVA-7B Evaluation (Full Dataset)

⚠️ **Estimated time: ~70–90 minutes for 70 images on CPU.**  
Results are checkpointed every 10 images. Do not interrupt if possible.

In [ ]:
OLLAMA_MODEL_LLAVA = "llava:7b-v1.6"
LLAVA7B_CSV = os.path.join(RESULTS_DIR, 'llava7b_results.csv')

# Remove existing (potentially corrupted) file before starting
if os.path.exists(LLAVA7B_CSV):
    os.remove(LLAVA7B_CSV)
    print("🗑️ Old results file removed — starting fresh")

PROMPT_LLAVA = """You are an expert in leather quality inspection.
Analyze this leather sample image carefully.

Answer these three questions:
1. DEFECT_PRESENT: Are there any visible defects on this leather surface? Answer only Yes or No.
2. DEFECT_TYPE: If yes, what type of defects do you see?
   Choose from: color, grain_off, folding_marks, pin_holes, cuts_scars, wrinkles, loose_grain, other.
   If no defects, write: none
3. SEVERITY: Rate the severity: None, Minor, or Major.

Format your answer exactly like this:
DEFECT_PRESENT: Yes/No
DEFECT_TYPE: type1, type2
SEVERITY: None/Minor/Major
DESCRIPTION: one sentence describing what you see"""

all_results = []

print(f"🚀 Running {OLLAMA_MODEL_LLAVA} on {len(df_work)} images")
print(f"   Estimated time: ~{len(df_work)*66/60:.0f} minutes")
print("   ⚠️ Do NOT interrupt — checkpointed every 10 images\n")

for idx, (_, row) in enumerate(tqdm(df_work.iterrows(), total=len(df_work), desc="LLaVA-7B")):
    start    = time.time()
    response = query_ollama(row['image_path_jpg'], PROMPT_LLAVA, OLLAMA_MODEL_LLAVA)
    elapsed  = time.time() - start
    parsed   = parse_llava_response(response)

    all_results.append({
        'filename'              : row['filename'],
        'leather_type'          : row['leather_type'],
        'gt_defect_present'     : row['defect_present'],
        'gt_defects'            : row['defects_raw'],
        'llava7b_defect_present': parsed['defect_present'],
        'llava7b_defect_type'   : parsed['defect_type'],
        'llava7b_severity'      : parsed['severity'],
        'llava7b_description'   : parsed['description'],
        'llava7b_raw_response'  : parsed['raw_response'],
        'processing_time_s'     : round(elapsed, 1)
    })

    # Safety checkpoint every 10 images
    if (idx + 1) % 10 == 0:
        pd.DataFrame(all_results).to_csv(LLAVA7B_CSV, index=False, encoding='utf-8-sig')
        print(f"   💾 [{idx+1}/{len(df_work)}] checkpoint saved")

# Final save
df_final = pd.DataFrame(all_results)
df_final.to_csv(LLAVA7B_CSV, index=False, encoding='utf-8-sig')
print(f"\n✅ Done! {len(df_final)} results saved to: {LLAVA7B_CSV}")

## Cell 12 — Final Comparison

In [ ]:
df_llava7b = pd.read_csv(os.path.join(RESULTS_DIR, 'llava7b_results.csv'))
df_llava7b = df_llava7b.drop_duplicates(subset='filename', keep='last')
df_llava7b_valid = df_llava7b[df_llava7b['llava7b_defect_present'].notna()].copy()

y_true_7b = df_llava7b_valid['gt_defect_present'].astype(bool).astype(int)
y_pred_7b = df_llava7b_valid['llava7b_defect_present'].astype(bool).astype(int)

acc_7b  = accuracy_score(y_true_7b, y_pred_7b)
prec_7b = precision_score(y_true_7b, y_pred_7b, zero_division=0)
rec_7b  = recall_score(y_true_7b, y_pred_7b, zero_division=0)
f1_7b   = f1_score(y_true_7b, y_pred_7b, zero_division=0)

print(f"LLaVA-7B: {len(df_llava7b_valid)} samples processed\n")
print("="*60)
print(f"  {'Metric':<12} {'CLIP':>10} {'Moondream':>12} {'LLaVA-7B':>12}")
print("="*60)
print(f"  {'Accuracy':<12} {acc*100:>9.1f}% {acc_l*100:>11.1f}% {acc_7b*100:>11.1f}%")
print(f"  {'Precision':<12} {prec*100:>9.1f}% {prec_l*100:>11.1f}% {prec_7b*100:>11.1f}%")
print(f"  {'Recall':<12} {rec*100:>9.1f}% {rec_l*100:>11.1f}% {rec_7b*100:>11.1f}%")
print(f"  {'F1-Score':<12} {f1*100:>9.1f}% {f1_l*100:>11.1f}% {f1_7b*100:>11.1f}%")
print("="*60)

avg_time_7b = pd.to_numeric(df_llava7b_valid['processing_time_s'], errors='coerce').mean()
print(f"\n  Average inference time per image:")
print(f"    CLIP      : ~0.5s")
print(f"    Moondream : ~5s")
print(f"    LLaVA-7B  : ~{avg_time_7b:.0f}s")

# Comparison figure
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

cm_7b = confusion_matrix(y_true_7b, y_pred_7b)
sns.heatmap(cm_7b, annot=True, fmt='d', cmap='Purples', ax=axes[0],
            xticklabels=['Pred: Clean', 'Pred: Defect'],
            yticklabels=['True: Clean', 'True: Defect'])
axes[0].set_title('LLaVA-7B — Confusion Matrix', fontsize=13, fontweight='bold')

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1']
clip_vals  = [acc*100, prec*100, rec*100, f1*100]
moon_vals  = [acc_l*100, prec_l*100, rec_l*100, f1_l*100]
llava_vals = [acc_7b*100, prec_7b*100, rec_7b*100, f1_7b*100]
x = np.arange(len(metrics_list))
w = 0.25

axes[1].bar(x - w, clip_vals,  w, label='CLIP',      color='#2196F3')
axes[1].bar(x,     moon_vals,  w, label='Moondream', color='#FF9800')
axes[1].bar(x + w, llava_vals, w, label='LLaVA-7B',  color='#9C27B0')
axes[1].set_xticks(x); axes[1].set_xticklabels(metrics_list)
axes[1].set_ylim(0, 110)
axes[1].set_ylabel('Value (%)')
axes[1].set_title('All 3 Models — Metric Comparison', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'final_comparison_3models.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save final summary table
final_table = pd.DataFrame({
    'Model'               : ['CLIP ViT-B/32', 'Moondream', 'LLaVA-7B-v1.6'],
    'Accuracy'            : [acc, acc_l, acc_7b],
    'Precision'           : [prec, prec_l, prec_7b],
    'Recall'              : [rec, rec_l, rec_7b],
    'F1'                  : [f1, f1_l, f1_7b],
    'Avg_time_per_image_s': [0.5, 5, avg_time_7b],
    'N_samples'           : [len(df_clip), len(df_llava_valid), len(df_llava7b_valid)]
})
final_table.to_csv(os.path.join(RESULTS_DIR, 'final_results_table.csv'), index=False)
print(f"\n✅ Final summary table saved to final_results_table.csv")
final_table

## Cell 13 — Error Analysis
Identifies samples where models disagree or fail, for use in the paper's error analysis section.

In [ ]:
df_merged = df_clip.merge(
    df_llava7b_valid[['filename', 'llava7b_defect_present', 'llava7b_defect_type',
                      'llava7b_severity', 'llava7b_description']],
    on='filename', how='inner'
)
df_merged['clip_correct']       = (df_merged['gt_defect_present'] == df_merged['clip_defect_predicted'])
df_merged['llava_correct']      = (df_merged['gt_defect_present'] == df_merged['llava7b_defect_present'].astype(bool))
df_merged['both_wrong']         = (~df_merged['clip_correct']) & (~df_merged['llava_correct'])
df_merged['clip_only_wrong']    = (~df_merged['clip_correct']) & df_merged['llava_correct']
df_merged['llava_only_wrong']   = df_merged['clip_correct'] & (~df_merged['llava_correct'])

print("📊 Error Analysis Summary:")
print(f"   Both correct        : {(df_merged['clip_correct'] & df_merged['llava_correct']).sum()}")
print(f"   Only CLIP wrong     : {df_merged['clip_only_wrong'].sum()}")
print(f"   Only LLaVA wrong    : {df_merged['llava_only_wrong'].sum()}")
print(f"   Both wrong          : {df_merged['both_wrong'].sum()}")

both_wrong = df_merged[df_merged['both_wrong']]
print(f"\n🔍 Samples where both models fail ({len(both_wrong)} images):")
for _, r in both_wrong.head(5).iterrows():
    print(f"   {r['filename']} | GT: {'defect' if r['gt_defect_present'] else 'clean'} | "
          f"CLIP: {r['clip_top_label']} | LLaVA: {r['llava7b_defect_type']} | "
          f"GT defects: {r['gt_defects']}")

df_merged.to_csv(os.path.join(RESULTS_DIR, 'full_analysis.csv'), index=False, encoding='utf-8-sig')
print(f"\n✅ Full analysis saved to full_analysis.csv")
print(f"\n📁 All results in: {RESULTS_DIR}")
print("   ├── labels_ground_truth.csv")
print("   ├── clip_results.csv")
print("   ├── llava_moondream_results.csv")
print("   ├── llava7b_results.csv")
print("   ├── full_analysis.csv")
print("   └── final_results_table.csv")

## Cell 14 — LLaVA-13B: Qualitative Test on Selected Samples
Tests a newer, larger LLaVA model on a small number of manually selected images for qualitative comparison.  
Change the filenames in `test_filenames` to test any samples.

In [ ]:
OLLAMA_MODEL_13B = "llava:13b"

# ► Change filenames to test any samples from the dataset
test_filenames = ['BO001.tif', 'BO005.tif', 'EL003.tif']

for fname in test_filenames:
    row = df_work[df_work['filename'] == fname]
    if len(row) == 0:
        print(f"❌ {fname} not found in df_work")
        continue
    row = row.iloc[0]

    print(f"\n{'='*50}")
    print(f"Image: {fname} | GT: {'defect' if row['defect_present'] else 'clean'}")
    start    = time.time()
    response = query_ollama(row['image_path_jpg'], PROMPT_LLAVA, OLLAMA_MODEL_13B)
    elapsed  = time.time() - start
    print(f"Time: {elapsed:.0f}s")
    print(f"Response:\n{response}")